In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip install xgboost==3.2.0
import xgboost
from xgboost import XGBClassifier
print(f"XGBoost: {xgboost.__version__}")

XGBoost: 3.2.0


In [14]:
import pandas as pd
import os
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score,confusion_matrix

In [15]:
import sys
import platform
import pandas as pd
import sklearn

print("Python Version :", sys.version)
print("OS Platform    :", platform.platform())
print("Pandas Version :", pd.__version__)
print("Scikit-learn Version :", sklearn.__version__)

Python Version : 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
OS Platform    : Linux-6.6.122+-x86_64-with-glibc2.39
Pandas Version : 2.2.3
Scikit-learn Version : 1.6.1


In [17]:
import pandas as pd
data = pd.read_csv('/content/Lung_Sound_975_Audio_3s_Segment_Features.csv')
data=data.drop(columns=['Location','Lung Sound ID','Gender'])
ran=42
data = data.sample(frac=1, random_state=ran)

n = len(data)
test_size = int(0.3 * n)
shift_size = int(0.1 * n)

All_Train, All_Test = [], []
for i in range(10):
    test_start = i * shift_size
    test_end = test_start + test_size
    if test_end <= n:
        test_set = data.iloc[test_start:test_end]
    else:
        test_set = pd.concat([data.iloc[test_start:], data.iloc[:test_end - n]])
    train_set = data.drop(test_set.index)
    All_Train.append(train_set)
    All_Test.append(test_set)



from sklearn.preprocessing import LabelEncoder

# Only Fold 1
i = 0

trainset = All_Train[i].copy()
testset = All_Test[i].copy()

categorical_columns = trainset.select_dtypes(include=['object']).columns
label_encoders = {}

print("\n========== Fold 1 ==========")

for col in categorical_columns:
    le = LabelEncoder()

    trainset[col] = le.fit_transform(trainset[col])
    testset[col] = le.transform(testset[col])

    label_encoders[col] = le

    # Print mapping
    print(f"\nColumn: {col}")
    for original, encoded in zip(le.classes_, le.transform(le.classes_)):
        print(f"{original} --> {encoded}")


========== Fold 1 ==========

Column: Lung Sound Type
Coarse Crackles --> 0
Fine Crackles --> 1
Normal --> 2
Pleural Rub --> 3
Rhonchi --> 4
Wheezing --> 5


In [18]:
# %%
import pandas as pd
import os
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score,confusion_matrix
import time
import shap
from matplotlib import rcParams
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

Optimizer="WOA-XGB"
all_results = []  # Store mean performance for each t
import numpy as np

#Finalized Parameters obtained from optimizatiion and voting
learning_rate = 0.009814178902222342
n_estimators = 289
max_depth = 12
min_child_weight = 3

Finalized_Feature= {0, 2, 4, 6, 7, 8}

print("Final Parameters")
print(f"learning_rate: {learning_rate}")
print(f"n_estimators: {n_estimators}")
print(f"max_depth: {max_depth}")
print(f"min_child_weight: {min_child_weight}")
print(f"Feature Number: {len(Finalized_Feature)}")
print(f"Finalized Features: {Finalized_Feature}")

final_model = XGBClassifier(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    min_child_weight=min_child_weight,
                    random_state=ran,
                )

Fold_result = []

for i in range(10):
    trainset = All_Train[i].copy()
    testset = All_Test[i].copy()
    categorical_columns = trainset.select_dtypes(include=['object']).columns
    label_encoders = {}
    for col in categorical_columns:
      le = LabelEncoder()
      trainset[col] = le.fit_transform(trainset[col])
      testset[col] = le.transform(testset[col])
      label_encoders[col] = le

    X_train = trainset.drop(['Lung Sound Type'], axis=1)
    Y_train = trainset['Lung Sound Type']
    X_test = testset.drop(['Lung Sound Type'], axis=1)
    Y_test = testset['Lung Sound Type']

    X_train_selected = X_train.iloc[:, list(Finalized_Feature)]
    X_test_selected = X_test.iloc[:, list(Finalized_Feature)]

    start_time = time.time()

    final_model.fit(X_train_selected, Y_train)
    train_time = time.time() - start_time

    preds_train = final_model.predict(X_train_selected)
    preds_test = final_model.predict(X_test_selected)

    accuracy_train = accuracy_score(Y_train, preds_train,)
    f1_score_train = f1_score(Y_train, preds_train,average='weighted')
    recall_score_train = recall_score(Y_train, preds_train,average='weighted')
    precision_score_train = precision_score(Y_train, preds_train,average='weighted')

    test_time = time.time()
    accuracy_test = accuracy_score(Y_test, preds_test)
    f1_score_test = f1_score(Y_test, preds_test,average='weighted')
    recall_score_test = recall_score(Y_test, preds_test,average='weighted')
    precision_score_test = precision_score(Y_test, preds_test,average='weighted')
    test_time = time.time() - test_time

    current_fold_info = {
        'Optimizer': Optimizer,
        'fold': i+1,
        'Train Accuracy': accuracy_train * 100,
        'Train F-score': f1_score_train * 100,
        'Train Precision Score': precision_score_train * 100,
        'Train Recall Score': recall_score_train * 100,
        'Test Accuracy': accuracy_test * 100,
        'Test F-score': f1_score_test * 100,
        'Test Precision Score': precision_score_test * 100,
        'Test Recall Score': recall_score_test * 100,
    }

    Fold_result.append(current_fold_info)

Fold_result_df = pd.DataFrame(Fold_result)
Fold_result_performance={
    'Train Accuracy': round(Fold_result_df['Train Accuracy'].mean(),2),
    'Train F-score': round(Fold_result_df['Train F-score'].mean(),2),
    'Train Precision': round(Fold_result_df['Train Precision Score'].mean(),2),
    'Train Recall': round(Fold_result_df['Train Recall Score'].mean(),2),
    'Test Accuracy': round(Fold_result_df['Test Accuracy'].mean(),2),
    'Test F-score': round(Fold_result_df['Test F-score'].mean(),2),
    'Test Precision': round(Fold_result_df['Test Precision Score'].mean(),2),
    'Test Recall': round(Fold_result_df['Test Recall Score'].mean(),2)
    }


Fold_result_performance_df=pd.DataFrame([Fold_result_performance])
Fold_result_performance_df

Final Parameters
learning_rate: 0.009814178902222342
n_estimators: 289
max_depth: 12
min_child_weight: 3
Feature Number: 6
Finalized Features: {0, 2, 4, 6, 7, 8}


,Train Accuracy,Train F-score,Train Precision,Train Recall,Test Accuracy,Test F-score,Test Precision,Test Recall
0,96.93,96.93,96.95,96.93,70.38,70.07,70.69,70.38
